In [ ]:
f_bar = (f * p_solve(params[0], params[1].map(jnp.exp), *params[2:], sampling_times).ys).integral()
x_neutral = np.interp(f_bar, f.vals, f.x)

# Find min and max observed affinity for each time point
min_x_observed = []
max_x_observed = []
for dist in affinity_dists.vals:
    # Find where the distribution is above some threshold
    nonzero_indices = np.where(dist > dist.max() * 1e-3)[0]  # adjust threshold as needed
    if len(nonzero_indices) > 0:
        min_x_observed.append(x[nonzero_indices[0]])
        max_x_observed.append(x[nonzero_indices[-1]])
    else:
        min_x_observed.append(x[0])  # fallback
        max_x_observed.append(x[-1])
min_x_observed = np.array(min_x_observed)
max_x_observed = np.array(max_x_observed)

plt.figure(figsize=(4, 3))

# Define colors matching the combined GC plot
color_map = {11: 'C0', 20: 'C1', 70: 'C2'}  # blue, orange, green
target_times = [11, 20, 70]

# Plot stacked histograms first (in background)
hist_base = -0.5
hist_height = 0.3

# Compute all histograms and find max total for normalization
hist_data = {}
for i, time in enumerate(sampling_times):
    if time not in target_times:
        continue
    # Get relative affinity distribution
    x_relative = x - x_neutral[i]
    mask = (x >= min_x_observed[i]) & (x <= max_x_observed[i])
    hist_data[time] = affinity_dists.vals[i][mask]

# Calculate max total for normalization
max_total = 0
for time in target_times:
    if time in hist_data:
        max_total = max(max_total, hist_data[time].max())

# Plot stacked histograms
bottom = None
for time in target_times:
    if time not in hist_data:
        continue

    i = list(sampling_times).index(time)
    x_relative = x - x_neutral[i]
    mask = (x >= min_x_observed[i]) & (x <= max_x_observed[i])
    x_masked = x_relative[mask]
    dist_masked = hist_data[time]

    # Normalize distribution
    dist_normalized = (dist_masked / max_total) * hist_height

    if bottom is None:
        bottom = np.full_like(dist_normalized, hist_base)
    else:
        # Interpolate bottom to match current x grid if needed
        bottom = np.interp(x_masked, x_prev, bottom_prev)

    top = bottom + dist_normalized

    plt.fill_between(x_masked, bottom, top, color=color_map[time], alpha=0.5, step='mid')

    # Save for next iteration
    bottom_prev = top
    x_prev = x_masked

# Plot curves
for i, time in enumerate(sampling_times):
    if time not in target_times:
        continue
    # Create mask for points between min and max observed affinity
    mask = (x >= min_x_observed[i]) & (x <= max_x_observed[i])
    x_plot = (x[mask] - x_neutral[i])
    f_plot = (f.vals[mask] - f_bar[i])

    plt.plot(x_plot, f_plot, lw=2, alpha=0.6, color=color_map[time],
             label=f"$t={int(time)}$, $x_0=${x_neutral[i]:.1f}")

plt.legend(loc="upper left", fontsize=8)
plt.axhline(0, color="k", ls="--", lw=1, alpha=0.4)
plt.axvline(0, color="k", ls="--", lw=1, alpha=0.4)
plt.xlabel(r"time-point-relative affinity ($x - x_0(t)$)")
plt.ylabel(r"growth rate ($f(x) - \bar{f}(t)$)")
plt.xlim(-2, 3)
plt.ylim(-0.6, 1.2)
plt.tight_layout()
print(output_dir)
plt.savefig(f"{output_dir}/pepsi-slices-with-affy-hists.pdf")
plt.show()